# 🌸 Clasificador de Flores — CNN desde Cero
### Proyecto Deep Learning — Reserva Botánica
---

## ⚙️ CELDA 0 — Configuración Global de Rutas
Ejecuta esta celda PRIMERO antes que cualquier otra.

In [ ]:
import os
import numpy as np
import tensorflow as tf

# Rutas del proyecto
PROJECT_DIR     = '/content/flores_project'
DATA_DIR        = os.path.join(PROJECT_DIR, 'flower_photos')
MODELS_DIR      = os.path.join(PROJECT_DIR, 'modelos_guardados')
BEST_MODEL_PATH = os.path.join(MODELS_DIR, 'mejor_modelo_flores.keras')

os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(MODELS_DIR,  exist_ok=True)

# Semillas de reproducibilidad
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Hiperparámetros globales
IMG_SIZE    = 128
BATCH_SIZE  = 32
NUM_CLASSES = 5
EPOCHS      = 60
VAL_SPLIT   = 0.2

print('=' * 55)
print('  CONFIGURACIÓN GLOBAL')
print('=' * 55)
print(f'  TensorFlow  : {tf.__version__}')
print(f'  Proyecto    : {PROJECT_DIR}')
print(f'  Dataset     : {DATA_DIR}')
print(f'  Modelos     : {MODELS_DIR}')
print(f'  IMG_SIZE    : {IMG_SIZE}x{IMG_SIZE}')
print(f'  BATCH_SIZE  : {BATCH_SIZE}')
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'  GPU         : ✅ {gpus[0].name}')
else:
    print('  GPU         : ⚠️  No detectada — activa GPU en Entorno de ejecución')
print('=' * 55)

---
## 📓 ETAPA 1 — Descarga y Exploración del Dataset

**¿Qué hacemos aquí?**
Descargamos el dataset `flower_photos` de Google/TensorFlow. Contiene **3.670 imágenes** de 5 especies de flores organizadas en carpetas. Cada carpeta es una clase: `daisy`, `dandelion`, `roses`, `sunflowers`, `tulips`.

**¿Por qué explorar antes de entrenar?**
Necesitamos conocer el balance de clases, tamaños de imagen y aspecto visual antes de diseñar el modelo.

In [ ]:
import urllib.request
import tarfile
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image

# --- Descarga ---
DATASET_URL   = 'http://download.tensorflow.org/example_images/flower_photos.tgz'
DOWNLOAD_PATH = os.path.join(PROJECT_DIR, 'flower_photos.tgz')

if not os.path.exists(DATA_DIR):
    print('⬇️  Descargando dataset...')
    urllib.request.urlretrieve(DATASET_URL, DOWNLOAD_PATH)
    print('📦  Extrayendo...')
    with tarfile.open(DOWNLOAD_PATH, 'r:gz') as tar:
        tar.extractall(PROJECT_DIR)
    print('✅  Listo!')
else:
    print('✅  Dataset ya existe.')

# --- Clases ---
CLASSES = sorted([d for d in os.listdir(DATA_DIR)
                  if os.path.isdir(os.path.join(DATA_DIR, d))])
print(f'\n🌸 Clases ({len(CLASSES)}): {CLASSES}')

# --- Conteo por clase ---
VALID_EXT = ('.jpg', '.jpeg', '.png')
counts    = {}
all_imgs  = []
for cls in CLASSES:
    p    = os.path.join(DATA_DIR, cls)
    imgs = [f for f in os.listdir(p) if f.lower().endswith(VALID_EXT)]
    counts[cls] = len(imgs)
    all_imgs.extend([os.path.join(p, f) for f in imgs])

total = sum(counts.values())
print(f'\n📊 Distribución:')
print(f'{"Clase":<15} {"Imgs":>8} {"Pct":>8}')
print('-' * 35)
for cls, cnt in counts.items():
    print(f'{cls:<15} {cnt:>8}   {cnt/total*100:>5.1f}%')
print(f'{"TOTAL":<15} {total:>8}')

# --- Inspección de tamaños ---
sample_paths = np.random.choice(all_imgs, size=100, replace=False)
widths, heights = [], []
for path in sample_paths:
    with Image.open(path) as img:
        w, h = img.size
        widths.append(w); heights.append(h)
print(f'\n🔍 Tamaños (muestra 100 imgs):')
print(f'   Ancho : min={min(widths)} max={max(widths)} prom={np.mean(widths):.0f}px')
print(f'   Alto  : min={min(heights)} max={max(heights)} prom={np.mean(heights):.0f}px')

# --- Visualización de ejemplos ---
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
fig.suptitle('Ejemplos del Dataset — 2 imágenes por clase', fontsize=14, fontweight='bold')
for col, cls in enumerate(CLASSES):
    cls_path = os.path.join(DATA_DIR, cls)
    imgs     = [f for f in os.listdir(cls_path) if f.lower().endswith(VALID_EXT)]
    sample   = np.random.choice(imgs, size=2, replace=False)
    for row in range(2):
        ax = axes[row][col]
        ax.imshow(mpimg.imread(os.path.join(cls_path, sample[row])))
        ax.set_title(cls, fontsize=11, fontweight='bold')
        ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'exploracion_dataset.png'), dpi=100)
plt.show()

# --- Distribución de clases ---
fig, ax = plt.subplots(figsize=(9, 5))
colors  = ['#FF9AA2','#FFB7B2','#FFDAC1','#E2F0CB','#B5EAD7']
bars    = ax.bar(counts.keys(), counts.values(), color=colors, edgecolor='black')
for bar, cnt in zip(bars, counts.values()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+10,
            str(cnt), ha='center', fontsize=11, fontweight='bold')
ax.axhline(y=total/len(CLASSES), color='red', linestyle='--',
           label=f'Promedio ({total//len(CLASSES)} imgs)')
ax.set_title('Distribución de imágenes por clase', fontweight='bold')
ax.set_xlabel('Clase'); ax.set_ylabel('Imágenes')
ax.legend(); plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'distribucion_clases.png'), dpi=100)
plt.show()
print('✅  Etapa 1 completada')

---
## 📓 ETAPA 2 — Preprocesamiento de Imágenes

**¿Qué hacemos aquí?**
Preparamos los datos para que la CNN los pueda consumir:
- **Normalización** → píxeles de [0,255] a [0,1] para estabilizar gradientes
- **Redimensionamiento** → todas las imágenes a 128×128 px (tamaño fijo)
- **Split 80/20** → 80% entrenamiento, 20% validación
- **Data Augmentation** → rotaciones, flips, zoom para reducir overfitting

**Importante:** el augmentation SOLO se aplica al set de entrenamiento, nunca a validación.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array

# --- Generador de ENTRENAMIENTO con Data Augmentation ---
train_datagen = ImageDataGenerator(
    rescale=1.0/255,
    rotation_range=30,
    horizontal_flip=True,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest',
    validation_split=VAL_SPLIT
)

# --- Generador de VALIDACIÓN (solo normalización) ---
val_datagen = ImageDataGenerator(
    rescale=1.0/255,
    validation_split=VAL_SPLIT
)

# --- Flujos de datos ---
train_generator = train_datagen.flow_from_directory(
    DATA_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical',
    subset='training', seed=SEED, shuffle=True
)
val_generator = val_datagen.flow_from_directory(
    DATA_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical',
    subset='validation', seed=SEED, shuffle=False
)

n_train = train_generator.samples
n_val   = val_generator.samples
print(f'\n📊 Split del dataset:')
print(f'   Entrenamiento : {n_train} imágenes ({n_train/(n_train+n_val)*100:.1f}%)')
print(f'   Validación    : {n_val} imágenes ({n_val/(n_train+n_val)*100:.1f}%)')
print(f'\n📌 Mapeo de clases: {train_generator.class_indices}')

# --- Verificar batch ---
batch_imgs, batch_labels = next(train_generator)
print(f'\n🔬 Verificación de batch:')
print(f'   Shape imágenes  : {batch_imgs.shape}  (esperado: (32,128,128,3))')
print(f'   Shape etiquetas : {batch_labels.shape} (esperado: (32,5))')
print(f'   Rango de píxeles: [{batch_imgs.min():.3f}, {batch_imgs.max():.3f}] (esperado: [0,1])')

# --- Visualizar Data Augmentation ---
sample_cls  = 'roses'
sample_file = os.listdir(os.path.join(DATA_DIR, sample_cls))[0]
sample_path = os.path.join(DATA_DIR, sample_cls, sample_file)
orig_img    = load_img(sample_path, target_size=(IMG_SIZE, IMG_SIZE))
img_arr     = img_to_array(orig_img).reshape((1, IMG_SIZE, IMG_SIZE, 3))

aug_gen = ImageDataGenerator(
    rotation_range=30, horizontal_flip=True, zoom_range=0.2,
    width_shift_range=0.1, height_shift_range=0.1,
    brightness_range=[0.7,1.3], fill_mode='nearest'
)
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
fig.suptitle('Data Augmentation — Original + 9 variaciones', fontsize=13, fontweight='bold')
axes[0][0].imshow(orig_img); axes[0][0].set_title('ORIGINAL', color='green', fontweight='bold'); axes[0][0].axis('off')
aug_iter = aug_gen.flow(img_arr, batch_size=1, seed=SEED)
positions = [(r,c) for r in range(2) for c in range(5)][1:]
for pos in positions:
    aug_img = next(aug_iter)[0].astype('uint8')
    axes[pos[0]][pos[1]].imshow(aug_img)
    axes[pos[0]][pos[1]].set_title('Aumentada', color='steelblue', fontsize=10)
    axes[pos[0]][pos[1]].axis('off')
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'data_augmentation.png'), dpi=100)
plt.show()
print('✅  Etapa 2 completada')

---
## 📓 ETAPA 3 — Construcción de la CNN desde Cero

**Arquitectura:** 4 bloques convolucionales + clasificador denso

Cada bloque sigue el patrón: `Conv2D → BatchNorm → ReLU → MaxPooling → Dropout`

| Bloque | Filtros | Salida | Aprende |
|--------|---------|--------|---------|
| 1 | 32 | 64×64×32 | Bordes, colores |
| 2 | 64 | 32×32×64 | Texturas, curvas |
| 3 | 128 | 16×16×128 | Formas de pétalos |
| 4 | 256 | 8×8×256 | Patrones complejos |

**GlobalAveragePooling** reemplaza a Flatten → 64x menos parámetros en la capa densa.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers, regularizers

L2_LAMBDA = 0.001

def conv_block(x, filters, dropout_rate=0.25):
    """Bloque: Conv2D → BatchNorm → ReLU → MaxPool → Dropout"""
    x = layers.Conv2D(filters, (3,3), padding='same', use_bias=False,
                      kernel_regularizer=regularizers.l2(L2_LAMBDA))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Dropout(dropout_rate)(x)
    return x

# --- Arquitectura ---
inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='input_imagen')
x = conv_block(inputs, 32,  dropout_rate=0.25)   # 128→64
x = conv_block(x,      64,  dropout_rate=0.25)   # 64→32
x = conv_block(x,      128, dropout_rate=0.30)   # 32→16
x = conv_block(x,      256, dropout_rate=0.30)   # 16→8
x = layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
x = layers.Dense(256, use_bias=False, kernel_regularizer=regularizers.l2(L2_LAMBDA), name='dense_256')(x)
x = layers.BatchNormalization(name='bn_dense')(x)
x = layers.Activation('relu', name='relu_dense')(x)
x = layers.Dropout(0.5, name='dropout_final')(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax', name='output')(x)

model = keras.Model(inputs=inputs, outputs=outputs, name='CNN_Flores')

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

total_p     = model.count_params()
trainable_p = sum([tf.size(w).numpy() for w in model.trainable_weights])
print(f'\n📊 Parámetros entrenables : {trainable_p:,}')
print(f'   Memoria aprox.         : {total_p*4/1024**2:.2f} MB')
print('✅  Etapa 3 completada')

---
## 📓 ETAPA 4 — Entrenamiento del Modelo

**Callbacks utilizados:**
- `ModelCheckpoint` → guarda el mejor modelo automáticamente
- `EarlyStopping` → detiene si val_loss no mejora en 12 épocas
- `ReduceLROnPlateau` → reduce learning rate si hay plateau

**Class weights:** compensan el desbalance del dataset penalizando más los errores en clases minoritarias.

In [ ]:
import time
from sklearn.utils.class_weight import compute_class_weight

# --- Callbacks ---
checkpoint_cb = keras.callbacks.ModelCheckpoint(
    filepath=BEST_MODEL_PATH, monitor='val_accuracy',
    save_best_only=True, save_weights_only=False,
    mode='max', verbose=1
)
early_stop_cb = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=12,
    restore_best_weights=True, mode='min', verbose=1
)
reduce_lr_cb = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.3, patience=5,
    min_lr=1e-7, mode='min', verbose=1
)

class EpochLogger(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        lr  = float(self.model.optimizer.learning_rate)
        ta  = logs.get('accuracy',0)*100
        va  = logs.get('val_accuracy',0)*100
        tl  = logs.get('loss',0)
        vl  = logs.get('val_loss',0)
        gap = ta - va
        st  = '⚠️ OVERFIT' if gap>15 else ('✅ Balance' if va>=ta else '📈 OK')
        print(f'  Ep {epoch+1:>3} | loss:{tl:.4f} acc:{ta:.1f}% | val_loss:{vl:.4f} val_acc:{va:.1f}% | lr:{lr:.1e} | {st}')

# --- Class Weights ---
train_labels        = train_generator.classes
cw_array            = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
class_weight_dict   = {i: w for i, w in enumerate(cw_array)}
print('📊 Class weights:')
for cls, (idx, w) in zip(CLASSES, class_weight_dict.items()):
    print(f'   [{idx}] {cls:<12} → {w:.4f}')

# --- Entrenamiento ---
print('\n🚀 Iniciando entrenamiento...')
t0 = time.time()

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=[checkpoint_cb, early_stop_cb, reduce_lr_cb, EpochLogger()],
    class_weight=class_weight_dict,
    verbose=0
)

elapsed    = time.time() - t0
epochs_run = len(history.history['loss'])
best_acc   = max(history.history['val_accuracy'])*100
best_ep    = np.argmax(history.history['val_accuracy'])+1

print(f'\n{"="*55}')
print(f'  ✅ ENTRENAMIENTO COMPLETADO')
print(f'  Épocas        : {epochs_run}/{EPOCHS}')
print(f'  Tiempo        : {elapsed/60:.1f} min')
print(f'  Mejor val_acc : {best_acc:.2f}% (época {best_ep})')
print(f'{"="*55}')

# --- Guardar modelo ---
try:
    model.save(BEST_MODEL_PATH)
    size_mb = os.path.getsize(BEST_MODEL_PATH)/1024**2
    print(f'\n💾 Modelo guardado: {BEST_MODEL_PATH} ({size_mb:.1f} MB)')
except Exception as e:
    print(f'⚠️  Error guardando: {e}')

# --- Curvas de aprendizaje ---
h  = history.history
ep = range(1, epochs_run+1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(ep, [v*100 for v in h['accuracy']], 'o-', color='#2196F3', label='Train')
axes[0].plot(ep, [v*100 for v in h['val_accuracy']], 's-', color='#F44336', label='Val')
axes[0].axvline(x=best_ep, color='#F44336', linestyle='--', alpha=0.5, label=f'Mejor: {best_acc:.1f}%')
axes[0].set_title('Accuracy por Época', fontweight='bold')
axes[0].set_xlabel('Época'); axes[0].set_ylabel('Accuracy (%)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(ep, h['loss'], 'o-', color='#2196F3', label='Train Loss')
axes[1].plot(ep, h['val_loss'], 's-', color='#F44336', label='Val Loss')
axes[1].set_title('Loss por Época', fontweight='bold')
axes[1].set_xlabel('Época'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Curvas de Aprendizaje — CNN Flores', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'curvas_entrenamiento.png'), dpi=100)
plt.show()
print('✅  Etapa 4 completada')

---
## 📓 ETAPA 5 — Evaluación del Modelo

**¿Qué analizamos?**
- **Accuracy y Loss** en el set de validación
- **Matriz de confusión** → qué clases confunde el modelo
- **Reporte de clasificación** → precision, recall y F1 por clase
- **Ejemplos de predicciones** correctas e incorrectas

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# --- Cargar mejor modelo ---
best_model = keras.models.load_model(BEST_MODEL_PATH)
print(f'✅ Modelo cargado desde: {BEST_MODEL_PATH}')

# --- Evaluación general ---
val_loss, val_acc = best_model.evaluate(val_generator, verbose=0)
print(f'\n📊 Resultados en Validación:')
print(f'   Loss     : {val_loss:.4f}')
print(f'   Accuracy : {val_acc*100:.2f}%')

# --- Predicciones ---
val_generator.reset()
y_pred_probs = best_model.predict(val_generator, verbose=1)
y_pred       = np.argmax(y_pred_probs, axis=1)
y_true       = val_generator.classes

# --- Reporte de clasificación ---
print(f'\n📋 Reporte por clase:')
print(classification_report(y_true, y_pred, target_names=CLASSES))

# --- Matriz de confusión ---
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES,
            linewidths=0.5, ax=ax)
ax.set_title('Matriz de Confusión', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Predicho', fontsize=12)
ax.set_ylabel('Real', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'matriz_confusion.png'), dpi=100)
plt.show()

# --- Visualizar predicciones correctas e incorrectas ---
correct_idx   = np.where(y_pred == y_true)[0]
incorrect_idx = np.where(y_pred != y_true)[0]

val_paths = [val_generator.filepaths[i] for i in range(len(val_generator.filepaths))]

fig, axes = plt.subplots(2, 5, figsize=(18, 8))
fig.suptitle('Predicciones del Modelo (Verde=Correcto | Rojo=Incorrecto)', fontsize=13, fontweight='bold')

for i, idx in enumerate(np.random.choice(correct_idx, 5, replace=False)):
    img = load_img(val_paths[idx], target_size=(IMG_SIZE, IMG_SIZE))
    axes[0][i].imshow(img)
    conf = y_pred_probs[idx][y_pred[idx]]*100
    axes[0][i].set_title(f'✅ {CLASSES[y_pred[idx]]}\n({conf:.1f}%)', color='green', fontsize=9, fontweight='bold')
    axes[0][i].axis('off')

for i, idx in enumerate(np.random.choice(incorrect_idx, 5, replace=False)):
    img = load_img(val_paths[idx], target_size=(IMG_SIZE, IMG_SIZE))
    axes[1][i].imshow(img)
    conf = y_pred_probs[idx][y_pred[idx]]*100
    axes[1][i].set_title(f'❌ Pred:{CLASSES[y_pred[idx]]}\nReal:{CLASSES[y_true[idx]]} ({conf:.1f}%)',
                         color='red', fontsize=9, fontweight='bold')
    axes[1][i].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'predicciones_ejemplos.png'), dpi=100)
plt.show()
print('✅  Etapa 5 completada')

---
## 📥 DESCARGA — Modelo y Archivos

In [ ]:
from google.colab import files
import zipfile

# Empaquetar todo en un ZIP
ZIP_PATH = '/content/flores_entregable.zip'
with zipfile.ZipFile(ZIP_PATH, 'w') as zf:
    zf.write(BEST_MODEL_PATH, 'mejor_modelo_flores.keras')
    for png in ['exploracion_dataset.png', 'distribucion_clases.png',
                'data_augmentation.png', 'curvas_entrenamiento.png',
                'matriz_confusion.png', 'predicciones_ejemplos.png']:
        p = os.path.join(PROJECT_DIR, png)
        if os.path.exists(p):
            zf.write(p, png)

size_zip = os.path.getsize(ZIP_PATH)/1024**2
print(f'📦 ZIP creado: {ZIP_PATH} ({size_zip:.1f} MB)')
print('⬇️  Iniciando descarga...')
files.download(ZIP_PATH)